# Annotated CIF tutorial — set up a pocket-conditioned design in a few steps

**Goal:** starting from a protein–ligand complex, build a single model-ready **annotated CIF** that asks RFProteina to design a new protein around the ligand. In a handful of steps we will make an input with:

- flexible **ligand** with some anchored atoms;
- **one pocket residue** fully fixed;
- two pocket residues with just tip atoms coordinate conditioned;
- mark **H-bond** donor/acceptor roles on the ligand's polar atoms;
- write the CIF and emit a small sweep of variants programmatically.

No JSON, no contig strings — just AtomWorks annotations on an atom array. See [`docs/annotated_cif_inputs.md`](../annotated_cif_inputs.md) for the full annotation reference.

In [ ]:
import numpy as np
from pathlib import Path

from atomworks.io import parse
from atomworks.io.utils.io_utils import to_cif_file
from atomworks.io.utils.annotator import ensure_annotations
from atomworks.io.utils.visualize import view
from atomworks.ml.conditions import C_SEQ, C_CRD, C_IDX
from atomworks.ml.utils.expandable_segment import insert_expandable_segment
from atomworks.enums import ChainType

from rfproteina.conditions import C_HB, C_IDXGRP, HBondEnum
from rfproteina.utils.parse_presets import DISTILLATION, ANNOTATED_CIF

# Point INPUT_CIF at your own protein–ligand complex. Default: a repo fixture.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / '.project-root').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
INPUT_CIF = REPO_ROOT / 'data/benchmarks/tip_atom_plinder_training_examples/5eni__1__1.A__1.B.cif'
OUT_DIR = REPO_ROOT / 'runs/annotated_cif_tutorial'
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Parse and subselect to the pocket

Parse the complex, then keep only the **ligand plus the three pocket residues nearest to it**. In this example, we fully fix `TYR 41` and will pin one tip atom each on `VAL 36` and `THR 87`.

In [ ]:
# Pocket residues nearest the ligand (chain, res_id), found offline.
scaffold_residue_ids = [41, 36, 87]
TIPS = {('1.A', 36): 'CG1', ('1.A', 87): 'OG1'}    # pin one sidechain tip atom each

aa = parse(
    str(INPUT_CIF),
    config=DISTILLATION  # see from rfproteina.utils.parse_presets for common configs you might want for parse
)['assemblies']['1'][0]
ensure_annotations(aa, 'is_ligand')  # check out atomworks.io.utils.annotator and atomworks.ml.utils.annotator for easily addable annotations!

keep_mask = aa.is_ligand | ((aa.chain_id == '1.A') & np.isin(aa.res_id, scaffold_residue_ids))
aa = aa[keep_mask]

print(aa.array_length(), 'atoms |', int(aa.is_ligand.sum()), 'ligand atoms')

In [ ]:
# Render inline: the ligand (sticks) surrounded by the three pocket residues.
view(aa)

## 2. Add an expandable segment

In [ ]:
aa = insert_expandable_segment(
    aa,
    seg_min=150,
    seg_max=200,
    new_chain=False,
    atom_idx=0,
    insert_before=True,
)

## 3. Apply the pocket conditioning

Three conditions define the motif:

- **Coordinates** (`C_CRD`) — fix the chlorine ligand atoms.
- **Sequence** (`C_SEQ`) — reveal identity for all scaffolded residues and the ligand.
- **Index** (`C_IDX` / `C_IDXGRP`) — mark the three motif residues **unindexed** (their position in the designed sequence is free) and put each in its **own index group** so they place independently.

In [ ]:
coord_cond_mask = (
    (aa.chain_id == '1.A') & (aa.res_id == 36) & (aa.atom_name == 'CG1') |  # first tip atom
    (aa.chain_id == '1.A') & (aa.res_id == 87) & (aa.atom_name == 'OG1') |  # second tip atom
    (aa.chain_id == '1.A') & (aa.res_id == 41)  # fully fixed residue (all atoms)
)
scaffold_residues = np.isin(aa.res_id, scaffold_residue_ids) & (aa.chain_id == '1.A')

# Coordinates: ligand + fully-fixed residue (all atoms) + the pinned tip atoms.
C_CRD.set_mask(aa, coord_cond_mask)

# Sequence: condition ligand and scaffolded residues (not expandable segment)
C_SEQ.set_mask(aa, scaffold_residues | aa.is_ligand)

# Index: unindex all three motif residues, then ensure each residue has it's own index group.
C_IDX.set_mask(aa, ~scaffold_residues)

# Because the unindexed residues have non-contiguous residue IDs, the default IDXGROUP will put each one in its own group.
# Here we are explicit in checking and resetting the IDXGROUP annotation manually. But, since the default behavior is what
# we want in this case, this step is not strictly necessary here.
groups = C_IDXGRP.annotation(aa, default="generate")
groups[aa.res_id == 36] = max(groups) + 1
groups[aa.res_id == 87] = max(groups) + 1
assert len(np.unique(groups[scaffold_residues])) == 3, "Expected three unique index groups for the three scaffold residues."
assert len(np.unique(groups[~scaffold_residues])) == 2, "Expected two unique index group for the ligand and expandable segment."
C_IDXGRP.set_annotation(aa, groups)  # Again, not strictly necessary here since we aren't changing from default

print('coord-fixed atoms:', int(C_CRD.mask(aa).sum()), '| unindexed atoms:', (~C_IDX.mask(aa)).sum())

## 4. Hydrogen-bond roles on the ligand

Request donor/acceptor roles on the ligand's polar atoms so the designed pocket satisfies them. `5QA` carries an amide plus a hydroxyl, which maps cleanly onto all three roles: the amide `N` **donates**, the carbonyl `O1` **accepts**, and the hydroxyl `O` does **both**.

In [ ]:
hbond = C_HB.default_annotation(aa).copy()
hbond[aa.is_ligand & (aa.atom_name == 'N')] = HBondEnum.DONOR      # amide N
hbond[aa.is_ligand & (aa.atom_name == 'O1')] = HBondEnum.ACCEPTOR  # carbonyl O
hbond[aa.is_ligand & (aa.atom_name == 'O')] = HBondEnum.BOTH       # hydroxyl O
C_HB.set_annotation(aa, hbond.astype(int))
print('H-bond atoms set:', int(C_HB.mask(aa).sum()))

## 5. Write the annotated CIF

Write with `save_standard_annotations=True` (all conditions embedded), then re-parse with the `ANNOTATED_CIF` preset — the exact path the model uses — to confirm every condition survives the round-trip.

In [ ]:
out_path = OUT_DIR / 'annotated_design.cif'
to_cif_file(
    aa,
    str(out_path),
    save_standard_annotations=True,  # this defaults to true, but is shown here for clarity since this is what saves all conditions to the file
    chain_disambiguation='chain_iid'
)

# if you want to load again, you can do so like this:
reloaded = parse(
    str(out_path),
    config=ANNOTATED_CIF  # this config ensures that all conditions are loaded from the file
)['assemblies']['1'][0]

## 6. Emit many variants programmatically

Coordinate conditioning takes its value straight from the atom coordinates, so we can sweep a pinned atom and re-save. Here we push the `THR 87` tip atom (`OG1`) directly away from the ligand in 0.1 Å steps, writing one CIF per step — a batch that probes how far the pocket tolerates that contact. Point `generate.py` at the directory to run them all.

In [ ]:
sweep = np.where((aa.chain_id == '1.A') & (aa.res_id == 87) & (aa.atom_name == 'OG1'))[0][0]
away = aa.coord[sweep] - aa.coord[aa.is_ligand].mean(axis=0)
away /= np.linalg.norm(away)  # unit vector pointing from the ligand toward the tip atom

variants_dir = OUT_DIR / 'variants'
variants_dir.mkdir(exist_ok=True)
base_xyz = aa.coord[sweep].copy()
for i in range(10):
    variant = aa.copy()
    variant.coord[sweep] = base_xyz + away * 0.1 * (i + 1)
    to_cif_file(variant, str(variants_dir / f'variant_{i:02d}.cif'),
                save_standard_annotations=True, chain_disambiguation='chain_iid')

## 7. Checking your work

If you load the saved annotated cif into pymol with the pymol plugin (see [`docs/annotated_cif_inputs.md`](../annotated_cif_inputs.md) for more info) you can visually inspect all the annotations, bonds, index groups, etc. to make sure you didn't make a mistake!